# Sentinel

## Create GroupKFold splits and save them

In [ ]:
import numpy as np
import json
import os
import pandas as pd

def build_group_kfold_split(groups, n_splits=5, seed=42):
    np.random.seed(seed)

    unique_groups = np.unique(groups)
    np.random.shuffle(unique_groups)

    fold_size = len(unique_groups) // n_splits

    folds = []

    for i in range(n_splits):
        test_groups = unique_groups[i * fold_size:(i + 1) * fold_size]
        train_groups = np.setdiff1d(unique_groups, test_groups)

        train_idx = np.where(np.isin(groups, train_groups))[0].tolist()
        test_idx = np.where(np.isin(groups, test_groups))[0].tolist()

        folds.append({
            "train_idx": train_idx,
            "test_idx": test_idx
        })

    return folds


# =========================
# Apply identical filtering as Dataset class
# (CRITICAL for consistency between training and split generation)
# =========================
csv_path = r"data/input/Sentinel-2.csv"
df = pd.read_csv(csv_path, encoding="gbk")

# -------------------------
# 1️⃣ NDWI filter
# -------------------------
ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
mask_ndwi = ndwi > 0.1

# -------------------------
# 2️⃣ clean filter
# -------------------------
x = df.iloc[:, :-2].values.astype(np.float32)
mask_clean = np.all(x >= 0, axis=1)

# -------------------------
# 3️⃣  mask
# -------------------------
mask = mask_ndwi & mask_clean

df = df[mask].reset_index(drop=True)

print("Filtered data size:", len(df))

# -------------------------
# 4️⃣ Construct group labels
# -------------------------
groups = np.arange(len(df))  # 每行一个样本

# -------------------------
# 5️⃣ KFold
# -------------------------
folds = build_group_kfold_split(groups, n_splits=5, seed=42)

# -------------------------
# 6️⃣ save
# -------------------------
save_path = r"data/input/Sentinel-2/Sentinel-2_splits.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

with open(save_path, "w") as f:
    json.dump(folds, f, indent=2)

print("Saved split to:", save_path)

Filtered data size: 1229
Saved split to: data/input/Sentinel-2/Sentinel-2_splits.json


## Shared loading and evaluation utilities

In [ ]:
import json
import numpy as np

def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]

    train_idx = np.array(split["train_idx"])
    test_idx = np.array(split["test_idx"])

    return train_idx, test_idx

## CSF-FPE Transformer（Continuous Spectral Fusion with Positional Encoding）

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F

DATA_DIR = r"data\input"
SPLIT_PATH = os.path.join(DATA_DIR, "Sentinel-2/Sentinel-2_splits.json")

# =========================
# ⭐ Dataset selection
# =========================
selected_datasets = ["sentinel"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
}

# =========================
# 0. Wavelength normalization
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# ⭐ split loader
# =========================
def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]
    return (
        np.array(split["train_idx"]),
        np.array(split["test_idx"])
    )

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),(740,15),
                     (783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        else:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=64, grid_size=16):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. Fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # fusion
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 4. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, d_model // 2),
            nn.SiLU(),
            nn.Linear(d_model // 2, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # =========================
        # 7. Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # -------------------------
        # 2. FiLM conditioning
        # -------------------------
        cond = torch.stack([
            dtype.float(),
            temp,
            wl.mean(dim=1).squeeze(-1),
            delta.mean(dim=1).squeeze(-1)
        ], dim=1)

        gamma, beta = self.film(cond).chunk(2, dim=1)
        z = gamma.unsqueeze(1) * z + beta.unsqueeze(1)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        z = torch.cat([t, z], dim=1)        # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ evaluation
# =========================
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            y_true.extend(batch["y"].cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, mae, r2


# =========================
# ⭐ train with early stopping
# =========================
def train():

    device = "cuda" if torch.cuda.is_available() else "cpu"

    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # global normalization
    # =========================
    all_x = []
    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)
    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    dataset = SpectralDataset(paths[0][1], paths[0][0], x_mean, x_std)

    # =========================
    # ⭐ 5-fold split
    # =========================
    folds = json.load(open(SPLIT_PATH))

    results = []

    for fold_id in range(5):

        print(f"\n================ FOLD {fold_id} ================")

        train_idx = np.array(folds[fold_id]["train_idx"])
        test_idx  = np.array(folds[fold_id]["test_idx"])

        train_ds = Subset(dataset, train_idx)
        test_ds  = Subset(dataset, test_idx)

        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=16)

        model = SpectralFusionModel().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=3e-4)

        # =========================
        # ⭐ early stopping
        # =========================
        best_loss = float("inf")
        patience = 50
        wait = 0
        best_model = None

        for epoch in range(500):

            model.train()
            total_loss = 0

            for batch in train_loader:
                pred = model(
                    batch["x"].to(device),
                    batch["lambda"].to(device),
                    batch["delta"].to(device),
                    batch["type"].to(device),
                    batch["temp"].to(device)
                )

                loss = r2_aware_loss(pred, batch["y"].to(device))

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item()

            # =========================
            # ⭐ validation = test here
            # =========================
            mse, rmse, mae, r2 = evaluate(model, test_loader, device)

            print(
                f"Epoch {epoch:03d} | "
                f"TrainLoss {total_loss:.4f} | "
                f"Test RMSE {rmse:.4f} R2 {r2:.4f}"
            )

            # =========================
            # ⭐ save best model
            # =========================
            if rmse < best_loss:
                best_loss = rmse
                best_model = copy.deepcopy(model.state_dict())
                wait = 0

                torch.save(
                    best_model,
                    os.path.join(DATA_DIR, f"Sentinel-2/CSF-PE Transformer/best_model_fold{fold_id}.pth")
                )
            else:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered")
                    break

        # =========================
        # ⭐ final evaluation
        # =========================
        model.load_state_dict(best_model)

        mse, rmse, mae, r2 = evaluate(model, test_loader, device)

        print(f"\n[FOLD {fold_id} FINAL]")
        print(f"MSE: {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
        print(f"R2: {r2:.4f}")

        results.append([mse, rmse, mae, r2])

    # =========================
    # ⭐ overall result
    # =========================
    results = np.array(results)

    print("\n================ FINAL 5-FOLD RESULT ================")
    print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
    print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
    print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
    print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

if __name__ == "__main__":
    train()


================ FOLD 0 ================
Epoch 000 | TrainLoss 2199.9563 | Test RMSE 4.1905 R2 -5.1542
Epoch 001 | TrainLoss 575.1251 | Test RMSE 1.7169 R2 -0.0331
Epoch 002 | TrainLoss 183.1323 | Test RMSE 1.6060 R2 0.0960
Epoch 003 | TrainLoss 168.7272 | Test RMSE 1.5085 R2 0.2025
Epoch 004 | TrainLoss 161.0324 | Test RMSE 1.4769 R2 0.2356
Epoch 005 | TrainLoss 158.1765 | Test RMSE 1.4707 R2 0.2420
Epoch 006 | TrainLoss 156.6007 | Test RMSE 1.5463 R2 0.1621
Epoch 007 | TrainLoss 154.5188 | Test RMSE 1.4691 R2 0.2436
Epoch 008 | TrainLoss 156.6370 | Test RMSE 1.4706 R2 0.2420
Epoch 009 | TrainLoss 157.0238 | Test RMSE 1.5053 R2 0.2059
Epoch 010 | TrainLoss 154.1811 | Test RMSE 1.4844 R2 0.2278
Epoch 011 | TrainLoss 158.6344 | Test RMSE 1.4823 R2 0.2299
Epoch 012 | TrainLoss 153.5004 | Test RMSE 1.4702 R2 0.2425
Epoch 013 | TrainLoss 155.4289 | Test RMSE 1.4579 R2 0.2551
Epoch 014 | TrainLoss 148.8857 | Test RMSE 1.5525 R2 0.1553
Epoch 015 | TrainLoss 149.7001 | Test RMSE 1.5150 R2 0.

grid_size =8
MSE  : 2.0034 ± 0.1544
RMSE : 1.4144 ± 0.0544
MAE  : 1.0878 ± 0.0333
R2   : 0.3180 ± 0.0459

## TabPFN

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from tabpfn import TabPFNRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
# Suppress warnings
warnings.filterwarnings("ignore")
# =========================
# path
# =========================
DATA_DIR = r"data/input"
SPLIT_PATH = os.path.join(DATA_DIR, "Sentinel-2/Sentinel-2_splits.json")

csv_path = os.path.join(DATA_DIR, "Sentinel-2.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# load split
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# data loading
# =========================
raw = pd.read_csv(csv_path, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = max(165, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "溶解氧"

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

# =========================
# group
# =========================
group_size = 1
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print("samples:", len(y), "features:", X.shape[1])
print("group samples:", num_samples)

# =========================
# load folds
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# result container
# =========================
results = []

# =========================
# 5-fold training 
# =========================
for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    # ===== Feature normalization =====
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # =========================
    # model
    # =========================
    model = TabPFNRegressor(
        n_estimators=32,
        device=device
    )

    # =========================
    # train
    # =========================
    model.fit(X_train, y_train)

    # =========================
    # predict
    # =========================
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    # =========================
    # evaluate
    # =========================
    train_metrics = evaluate(y_train, pred_train)
    test_metrics = evaluate(y_test, pred_test)

    print(f"[Train] MSE={train_metrics[0]:.4f} RMSE={train_metrics[1]:.4f} MAE={train_metrics[2]:.4f} R2={train_metrics[3]:.4f}")
    print(f"[Test ] MSE={test_metrics[0]:.4f}  RMSE={test_metrics[1]:.4f}  MAE={test_metrics[2]:.4f} R2={test_metrics[3]:.4f}")

    # =========================
    # save fold result
    # =========================
    results.append(test_metrics)

    # TabPFN没有可训练权重，这里保存预测结果
    save_dir = os.path.join(DATA_DIR, "Sentinel-2/TabPFN")
    os.makedirs(save_dir, exist_ok=True)

    np.save(os.path.join(save_dir, f"fold_{fold_id}_pred.npy"), pred_test)

# =========================
# final summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
samples: 1764 features: 11
group samples: 1764

================ FOLD 0 ================
[Train] MSE=1.1004 RMSE=1.0490 MAE=0.6773 R2=0.6661
[Test ] MSE=1.9866  RMSE=1.4095  MAE=0.9954 R2=0.4615

================ FOLD 1 ================
[Train] MSE=1.0819 RMSE=1.0401 MAE=0.6923 R2=0.6807
[Test ] MSE=1.9092  RMSE=1.3817  MAE=0.9866 R2=0.4298

================ FOLD 2 ================
[Train] MSE=1.1232 RMSE=1.0598 MAE=0.6903 R2=0.6810
[Test ] MSE=1.7549  RMSE=1.3247  MAE=0.9468 R2=0.3763

================ FOLD 3 ================
[Train] MSE=1.1932 RMSE=1.0924 MAE=0.7226 R2=0.6398
[Test ] MSE=2.2693  RMSE=1.5064  MAE=1.0421 R2=0.3759

================ FOLD 4 ================
[Train] MSE=1.1455 RMSE=1.0703 MAE=0.7118 R2=0.6599
[Test ] MSE=1.8925  RMSE=1.3757  MAE=0.9410 R2=0.4477

================ FINAL 5-FOLD RESULT ================
MSE  : 1.9625 ± 0.1706
RMSE : 1.3996 ± 0.0600
MAE  : 0.9824 ± 0.0367
R2   : 0.4182 ± 0.0358


================ FINAL 5-FOLD RESULT ================
MSE  : 2.0103 ± 0.1947
RMSE : 1.4162 ± 0.0677
MAE  : 0.9985 ± 0.0655
R2   : 0.4035 ± 0.0383

## Transformer

In [5]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# path
# =========================
DATA_DIR = r"data/input"
CSV_PATH = os.path.join(DATA_DIR, "Sentinel-2.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "Sentinel-2/Sentinel-2_splits.json")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# load split
# =========================
def load_folds(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# Transformer baseline model
# =========================
class TransformerBaseline(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2):
        super().__init__()

        self.embed = nn.Linear(1, d_model)

        self.pos = nn.Parameter(torch.randn(1, 200, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (B, N)
        x = x.unsqueeze(-1)
        x = self.embed(x)

        N = x.size(1)
        x = x + self.pos[:, :N, :].to(x.device)

        x = self.encoder(x)

        x = x.mean(dim=1)

        return self.head(x).squeeze()

# =========================
# load dataset
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = min(165, raw.shape[1])
target_col = "溶解氧"

X_df = raw.iloc[:, :max_feat].copy()

if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print("Samples:", len(y), "Features:", X.shape[1])

# =========================
# group
# =========================
group_size = 1
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

print("Group samples:", num_samples)

# =========================
# load folds
# =========================
folds = load_folds(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # tensor
    # =========================
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    # =========================
    # model
    # =========================
    model = TransformerBaseline().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # =========================
    # training loop
    # =========================
    best_loss = float("inf")
    patience = 50
    wait = 0

    for epoch in range(500):

        model.train()

        pred = model(X_train)
        loss = loss_fn(pred, y_train)

        opt.zero_grad()
        loss.backward()
        opt.step()

        # early stopping
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_model = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # =========================
    # evaluation
    # =========================
    model.load_state_dict(best_model)

    model.eval()
    with torch.no_grad():
        pred = model(X_test).cpu().numpy()

    mse, rmse, mae, r2 = evaluate(y_test, pred)

    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

    results.append([mse, rmse, mae, r2])

    # =========================
    # save
    # =========================
    save_dir = os.path.join(DATA_DIR, "Sentinel-2/Transformer")
    os.makedirs(save_dir, exist_ok=True)

    np.save(os.path.join(save_dir, f"fold_{fold_id}_pred.npy"), pred)
    torch.save(best_model, os.path.join(save_dir, f"fold_{fold_id}.pth"))

# =========================
# final result
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
Samples: 1764 Features: 11
Group samples: 1764

================ FOLD 0 ================
MSE=2.5462, RMSE=1.5957, MAE=1.1906, R2=0.3098

================ FOLD 1 ================
MSE=2.4506, RMSE=1.5655, MAE=1.1830, R2=0.2681

================ FOLD 2 ================
MSE=2.1739, RMSE=1.4744, MAE=1.0457, R2=0.2274

================ FOLD 3 ================
MSE=3.7767, RMSE=1.9434, MAE=1.5379, R2=-0.0386

================ FOLD 4 ================
MSE=3.2359, RMSE=1.7989, MAE=1.3874, R2=0.0557

================ FINAL 5-FOLD RESULT ================
MSE  : 2.8367 ± 0.5858
RMSE : 1.6756 ± 0.1708
MAE  : 1.2689 ± 0.1731
R2   : 0.1645 ± 0.1333


================ FINAL 5-FOLD RESULT ================
MSE  : 2.8918 ± 0.2625
RMSE : 1.6987 ± 0.0781
MAE  : 1.2775 ± 0.0871
R2   : 0.1414 ± 0.0608

## Residual MLP

In [6]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# path
# =========================
DATA_DIR = r"data/input"
CSV_PATH = os.path.join(DATA_DIR, "Sentinel-2.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "Sentinel-2/Sentinel-2_splits.json")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# load folds
# =========================
def load_folds(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# MLP model
# =========================
class ResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)
        self.relu = nn.ReLU()

        # 如果维度不同，用 projection
        self.proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        return self.relu(self.fc(x) + self.proj(x))


class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()

        self.block1 = ResidualBlock(in_dim, 256)
        self.block2 = ResidualBlock(256, 128)
        self.block3 = ResidualBlock(128, 64)
        self.block4 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return x.squeeze()

# =========================
# load dataset
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")

raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = min(165, raw.shape[1])
target_col = "溶解氧"

X_df = raw.iloc[:, :max_feat].copy()

if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values.astype(np.float32)
y = data.iloc[:, -1].values.astype(np.float32)

print("Samples:", len(y), "Features:", X.shape[1])

# =========================
# group
# =========================
group_size = 1
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

print("Group samples:", num_samples)

# =========================
# folds
# =========================
folds = load_folds(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # tensor
    # =========================
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    # =========================
    # model
    # =========================
    model = MLP(X.shape[1]).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # =========================
    # training loop
    # =========================
    best_loss = float("inf")
    patience = 50
    wait = 0

    for epoch in range(500):

        model.train()

        pred = model(X_train)
        loss = loss_fn(pred, y_train)

        opt.zero_grad()
        loss.backward()
        opt.step()

        # early stopping
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_model = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # =========================
    # evaluation
    # =========================
    model.load_state_dict(best_model)

    model.eval()
    with torch.no_grad():
        pred = model(X_test).cpu().numpy()

    mse, rmse, mae, r2 = evaluate(y_test, pred)

    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

    results.append([mse, rmse, mae, r2])

    # =========================
    # save
    # =========================
    save_dir = os.path.join(DATA_DIR, "Sentinel-2/MLP")
    os.makedirs(save_dir, exist_ok=True)

    np.save(os.path.join(save_dir, f"fold_{fold_id}_pred.npy"), pred)
    torch.save(best_model, os.path.join(save_dir, f"fold_{fold_id}.pth"))

# =========================
# final result
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
Samples: 1764 Features: 11
Group samples: 1764

================ FOLD 0 ================
MSE=2.4714, RMSE=1.5721, MAE=1.1305, R2=0.3300

================ FOLD 1 ================
MSE=2.7741, RMSE=1.6656, MAE=1.2617, R2=0.1715

================ FOLD 2 ================
MSE=2.2112, RMSE=1.4870, MAE=1.1429, R2=0.2141

================ FOLD 3 ================
MSE=2.8059, RMSE=1.6751, MAE=1.1766, R2=0.2283

================ FOLD 4 ================
MSE=2.7451, RMSE=1.6568, MAE=1.1854, R2=0.1989

================ FINAL 5-FOLD RESULT ================
MSE  : 2.6015 ± 0.2287
RMSE : 1.6113 ± 0.0722
MAE  : 1.1794 ± 0.0459
R2   : 0.2286 ± 0.0541


================ FINAL 5-FOLD RESULT ================
MSE  : 2.6345 ± 0.3561
RMSE : 1.6191 ± 0.1145
MAE  : 1.1937 ± 0.0952
R2   : 0.2188 ± 0.0889

In [8]:
results
# 保留4位小数，以tab分隔，复制到剪贴板
df.to_clipboard(index=False, header=False, sep='\t', float_format='%.4f')

array([[2.4714222 , 1.57207576, 1.13046515, 0.33003479],
       [2.77410817, 1.66556542, 1.26167953, 0.17148447],
       [2.21118498, 1.48700537, 1.14290726, 0.21412283],
       [2.80589962, 1.67508197, 1.17662752, 0.22834671],
       [2.74508357, 1.65682937, 1.18543971, 0.19890988]])

## XGBoost

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import copy

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold
import xgboost as xgb

# =========================
# path config
# =========================
DATA_DIR = r"\data\input"
CSV_PATH = os.path.join(DATA_DIR, "Sentinel-2.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "Sentinel-2/Sentinel-2_splits.json")

SAVE_DIR = os.path.join(DATA_DIR, "Sentinel-2/XGBoost")
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# load split
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# evaluation
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# load data
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip()

# feature selection
max_feat = min(165, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "溶解氧"

if target_col not in raw.columns:
    raise ValueError(f"Target column {target_col} not found")

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print(f"Samples: {len(y)}, Features: {X.shape[1]}")

# =========================
# group definition (same as PI-Transformer)
# =========================
group_size = 1
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print(f"Grouped samples: {num_samples}")

# =========================
# load folds
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # model
    # =========================
    model = xgb.XGBRegressor(
        n_estimators=2000,
        max_depth=1,
        learning_rate=0.01,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=1234
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # =========================
    # prediction
    # =========================
    y_pred = model.predict(X_test)

    mse, rmse, mae, r2 = evaluate(y_test, y_pred)

    print(f"Fold {fold_id} | MSE: {mse:.4f} RMSE: {rmse:.4f} MAE: {mae:.4f} R2: {r2:.4f}")

    # =========================
    # save model
    # =========================
    model_path = os.path.join(SAVE_DIR, f"xgb_fold{fold_id}.json")
    model.save_model(model_path)

    print(f"Saved: {model_path}")

    results.append([mse, rmse, mae, r2])

# =========================
# summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE: {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2  : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Samples: 1764, Features: 11
Grouped samples: 1764

================ FOLD 0 ================
Fold 0 | MSE: 2.2938 RMSE: 1.5145 MAE: 1.0927 R2: 0.3072
Saved: E:\LYQ\work\250916remote_model\data\input\Sentinel-2/XGBoost\xgb_fold0.json

================ FOLD 1 ================
Fold 1 | MSE: 2.8704 RMSE: 1.6942 MAE: 1.2681 R2: 0.2463
Saved: E:\LYQ\work\250916remote_model\data\input\Sentinel-2/XGBoost\xgb_fold1.json

================ FOLD 2 ================
Fold 2 | MSE: 2.4921 RMSE: 1.5786 MAE: 1.2076 R2: 0.2250
Saved: E:\LYQ\work\250916remote_model\data\input\Sentinel-2/XGBoost\xgb_fold2.json

================ FOLD 3 ================
Fold 3 | MSE: 2.3314 RMSE: 1.5269 MAE: 1.0995 R2: 0.2610
Saved: E:\LYQ\work\250916remote_model\data\input\Sentinel-2/XGBoost\xgb_fold3.json

================ FOLD 4 ================
Fold 4 | MSE: 2.3389 RMSE: 1.5294 MAE: 1.1413 R2: 0.3033
Saved: E:\LYQ\work\250916remote_model\data\input\Sentinel-2/XGBoost\xgb_fold4.json

================ FINAL 5-FOLD RESULT ==

================ FINAL 5-FOLD RESULT ================
MSE : 2.4653 ± 0.2136
RMSE: 1.5687 ± 0.0665
MAE : 1.1619 ± 0.0670
R2  : 0.2686 ± 0.0321

## Random Forest

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import copy
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# path config
# =========================
DATA_DIR = r"data\input"
CSV_PATH = os.path.join(DATA_DIR, "Sentinel-2.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "Sentinel-2/Sentinel-2_splits.json")

SAVE_DIR = os.path.join(DATA_DIR, "Sentinel-2/RandomForest")
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# load split
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# evaluation
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# load data
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip()

# =========================
# feature / target
# =========================
max_feat = min(165, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "溶解氧"

if target_col not in raw.columns:
    raise ValueError(f"Target column {target_col} not found")

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print(f"Samples: {len(y)}, Features: {X.shape[1]}")

# =========================
# group definition (same as PI-Transformer)
# =========================
group_size = 1
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print(f"Grouped samples: {num_samples}")

# =========================
# load splits
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # Random Forest model
    # =========================
    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=True,
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mse, rmse, mae, r2 = evaluate(y_test, y_pred)

    print(f"Fold {fold_id} | MSE: {mse:.4f} RMSE: {rmse:.4f} MAE: {mae:.4f} R2: {r2:.4f}")

    # =========================
    # save model
    # =========================
    model_path = os.path.join(SAVE_DIR, f"rf_fold{fold_id}.pkl")
    joblib.dump(model, model_path)

    print(f"Saved: {model_path}")

    results.append([mse, rmse, mae, r2])

# =========================
# summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE: {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2  : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Samples: 1764, Features: 11
Grouped samples: 1764

================ FOLD 0 ================
Fold 0 | MSE: 2.1871 RMSE: 1.4789 MAE: 1.0341 R2: 0.3394
Saved: data\input\Sentinel-2/RandomForest\rf_fold0.pkl

================ FOLD 1 ================
Fold 1 | MSE: 2.7190 RMSE: 1.6489 MAE: 1.2121 R2: 0.2861
Saved: data\input\Sentinel-2/RandomForest\rf_fold1.pkl

================ FOLD 2 ================
Fold 2 | MSE: 2.2978 RMSE: 1.5159 MAE: 1.1001 R2: 0.2854
Saved: data\input\Sentinel-2/RandomForest\rf_fold2.pkl

================ FOLD 3 ================
Fold 3 | MSE: 2.1538 RMSE: 1.4676 MAE: 1.0562 R2: 0.3173
Saved: data\input\Sentinel-2/RandomForest\rf_fold3.pkl

================ FOLD 4 ================
Fold 4 | MSE: 2.1843 RMSE: 1.4779 MAE: 1.0578 R2: 0.3494
Saved: data\input\Sentinel-2/RandomForest\rf_fold4.pkl

================ FINAL 5-FOLD RESULT ================
MSE : 2.3084 ± 0.2111
RMSE: 1.5178 ± 0.0676
MAE : 1.0921 ± 0.0637
R2  : 0.3155 ± 0.0264


================ FINAL 5-FOLD RESULT ================
MSE : 2.3084 ± 0.2111
RMSE: 1.5178 ± 0.0676
MAE : 1.0921 ± 0.0637
R2  : 0.3155 ± 0.0264